# FunctorFlow Wiki-2 KET Language Model

This notebook mirrors the PTB path, but switches to Wiki-2 so the same
FunctorFlow-backed KET family can be exercised on a second standard language
modeling benchmark using code that stays entirely inside `FunctorFlow/`.


In [ ]:
from pathlib import Path
import sys

def _bootstrap_functorflow() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        package_root = candidate / "FunctorFlow"
        if (package_root / "__init__.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError(
        "Could not find the FunctorFlow package. Run this notebook from the repo "
        "root or from FunctorFlow/notebooks."
    )

REPO_ROOT = _bootstrap_functorflow()
print(f"FunctorFlow repo root: {REPO_ROOT}")


In [ ]:
from FunctorFlow.ket_lm import (
    FunctorFlowKETLanguageModel,
    KETLanguageModelConfig,
    estimate_perplexity,
    load_word_language_modeling_corpus,
    pick_device,
    train_language_model,
)

device = pick_device("cuda")
corpus = load_word_language_modeling_corpus("wiki-2")
config = KETLanguageModelConfig.historical_wiki2_reference()
corpus.name, corpus.vocab_size, len(corpus.train_ids), len(corpus.valid_ids), config


## Build The Model

This uses the Wiki-2 historical-reference preset so PTB and Wiki-2 are now
driven by one shared FunctorFlow model surface.


In [ ]:
model = FunctorFlowKETLanguageModel(
    corpus.vocab_size,
    config=config,
)
model


## Train Briefly On Wiki-2

For tutorial iteration we keep the run short. The main point here is to verify
that the FunctorFlow KET code path transfers cleanly across benchmarks without
leaving the `FunctorFlow/` package boundary.


In [ ]:
history = train_language_model(
    model,
    corpus,
    steps=50,
    block_size=128,
    batch_size=16,
    lr=2e-3,
    device=device,
)
history


In [ ]:
valid_ppl = estimate_perplexity(
    model,
    corpus.valid_ids,
    block_size=128,
    batch_size=16,
    device=device,
)
valid_ppl


## Next Step

With PTB and Wiki-2 both rendered through FunctorFlow, the next move is to tune
one shared KET family more seriously and package the `FunctorFlow/` directory as
the standalone GitHub tutorial repo.
